# Overture — attribute pushdown with `where=` (live)

The bbox limits *where* you fetch; `where=` limits *which rows*, pushed down to the GeoParquet on S3 via DuckDB so only matching features leave the bucket. Here we keep only high-confidence places and a narrow set of columns (`id`/`sources` are retained automatically so the per-row `license_id` still works).

## Setup

Imports and the area of interest. `earthlens` provides the unified `EarthLens` entry point;
`geopandas` reads the GeoParquet that Overture writes.

In [ ]:
import geopandas as gpd
from earthlens.earthlens import EarthLens

### Area of interest

A narrow bounding box over a single Times Square block, plus the output directory `download()`
writes into.

In [ ]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

## Fetch with attribute pushdown

Build the Overture request first. The bbox limits *where* you fetch; `where=` limits *which rows*,
pushed down to the GeoParquet on S3 via DuckDB so only matching features leave the bucket. We keep
only high-confidence places (`confidence > 0.95`) and a narrow set of columns (`id`/`sources` are
retained automatically so the per-row `license_id` still works).

In [ ]:
overture = EarthLens(
    data_source="overture",
    dataset="places",
    variables=[],
    aoi=[LON_LIM[0], LAT_LIM[0], LON_LIM[1], LAT_LIM[1]],
    path=OUT,
    where="confidence > 0.95",  # pushed down to S3
    columns=["names", "confidence"],  # narrow projection
)

Run the download. `download()` returns the list of written GeoParquet paths.

In [ ]:
paths = overture.download()
paths

### Verify the pushdown

Read the written parquet back and confirm every row satisfies the `where=` predicate, with the
per-row `license_id` populated.

In [ ]:
gdf = gpd.read_parquet(paths[0])
print(len(gdf), 'high-confidence places')
print('all match the predicate:', bool((gdf['confidence'] > 0.95).all()))
gdf[['confidence', 'license_id']].head()

`where=` is raw SQL evaluated by DuckDB against the parquet schema, so you can filter nested fields too — e.g. `categories.primary = 'restaurant'` for places, or `height > 10` for buildings.